# Первая версия (во всех окнах):

In [ ]:
import winrawin
import tkinter as tk
import keyboard
import time
from threading import Timer

# Словарь для хранения информации об устройствах
device_map = {}
device_counter = 1

# Переменные для отслеживания состояния движения
movement_state = {
    'mouse1': {
        'is_moving_forward': False,   # Движение вверх (W)
        'is_moving_backward': False,  # Движение вниз (S)
        'last_move_time': 0,
        'last_y_position': 0,         # Для определения направления
        'w_pressed': False,
        's_pressed': False
    },
     'mouse2': {
        'is_moving_right': False,   # Движение вправо (D)
        'is_moving_left': False,  # Движение влево (A)
        'last_move_time': 0,
        'last_x_position': 0,         # Для определения направления
        'd_pressed': False,
        'a_pressed': False
    }
}

# Таймер для определения остановки движения
STOP_DELAY = 0.15  # Немного увеличим задержку для более плавного управления
MOVEMENT_THRESHOLD = 3  # Порог движения для определения направления (пиксели)

def list_mice_devices():
    """Выводит список всех подключенных мышей"""
    global device_map, device_counter
    
    print("🔍 Поиск подключенных мышей...\n")
    
    devices = winrawin.list_devices()
    mice_handles = []
    
    for device in devices:
        if isinstance(device, winrawin.Mouse):
            print(f"🖱️ Мышь #{device_counter}:")
            print(f"   Тип: {device.mouse_type}")
            print(f"   Handle: {device.handle}\n")
            
            device_map[device.handle] = {
                'id': device_counter,
                'type': device.mouse_type
            }
            
            mice_handles.append(device.handle)
            device_counter += 1
    
    return mice_handles

def press_w():
    """Нажимает и удерживает клавишу W (вперёд)"""
    if not movement_state['mouse1']['w_pressed']:
        # Если была нажата S, сначала отпускаем её
        if movement_state['mouse1']['s_pressed']:
            keyboard.release('s')
            movement_state['mouse1']['s_pressed'] = False
        
        keyboard.press('w')
        movement_state['mouse1']['w_pressed'] = True
        print("▶️ W НАЖАТА (движение ВПЕРЁД)")

def press_s():
    """Нажимает и удерживает клавишу S (назад)"""
    if not movement_state['mouse1']['s_pressed']:
        # Если была нажата W, сначала отпускаем её
        if movement_state['mouse1']['w_pressed']:
            keyboard.release('w')
            movement_state['mouse1']['w_pressed'] = False
        
        keyboard.press('s')
        movement_state['mouse1']['s_pressed'] = True
        print("▶️ S НАЖАТА (движение НАЗАД)")

def press_d():
    """Нажимает и удерживает клавишу D (вправо)"""
    if not movement_state['mouse2']['d_pressed']:
        # Если была нажата A, сначала отпускаем её
        if movement_state['mouse2']['a_pressed']:
            keyboard.release('a')
            movement_state['mouse2']['a_pressed'] = False
        
        keyboard.press('d')
        movement_state['mouse2']['d_pressed'] = True
        print("▶️ D НАЖАТА (движение ВПРАВО)")

def press_a():
    """Нажимает и удерживает клавишу A (влево)"""
    if not movement_state['mouse2']['a_pressed']:
        # Если была нажата D, сначала отпускаем её
        if movement_state['mouse2']['d_pressed']:
            keyboard.release('d')
            movement_state['mouse1']['d_pressed'] = False
        
        keyboard.press('a')
        movement_state['mouse2']['a_pressed'] = True
        print("▶️ A НАЖАТА (движение ВЛЕВО)")

def release_mouse1():
    """Отпускает все клавиши"""
    if movement_state['mouse1']['w_pressed']:
        keyboard.release('w')
        movement_state['mouse1']['w_pressed'] = False
        print("⏹️ W ОТПУЩЕНА")
    
    if movement_state['mouse1']['s_pressed']:
        keyboard.release('s')
        movement_state['mouse1']['s_pressed'] = False
        print("⏹️ S ОТПУЩЕНА")

def release_mouse2():
    """Отпускает все клавиши"""
    if movement_state['mouse2']['d_pressed']:
        keyboard.release('d')
        movement_state['mouse2']['d_pressed'] = False
        print("⏹️ D ОТПУЩЕНА")
    
    if movement_state['mouse2']['a_pressed']:
        keyboard.release('a')
        movement_state['mouse2']['a_pressed'] = False
        print("⏹️ A ОТПУЩЕНА")

def check_stop_moving():
    """Проверяет, нужно ли отпустить клавиши из-за остановки мыши"""
    current_time = time.time()
    
    # Если прошло больше STOP_DELAY с последнего движения
    if current_time - movement_state['mouse1']['last_move_time'] > STOP_DELAY:
        movement_state['mouse1']['is_moving_forward'] = False
        movement_state['mouse1']['is_moving_backward'] = False
        release_mouse1()
    if current_time - movement_state['mouse2']['last_move_time'] > STOP_DELAY:
        movement_state['mouse2']['is_moving_right'] = False
        movement_state['mouse2']['is_moving_left'] = False
        release_mouse2()

def handle_mouse_event(event: winrawin.RawInputEvent):
    """Обрабатывает события от мышей"""
    
    if event.event_type == 'move':
        device_handle = event.device.handle
        
        if device_handle in device_map:
            device_id = device_map[device_handle]['id']
            
            if device_id == 1:
                # Получаем движение по Y (вверх/вниз)
                y_movement = 0
                
                # В winrawin движение может быть в разных атрибутах
                # Пробуем получить из разных возможных мест
                if hasattr(event, 'y'):
                    y_movement = event.y
                elif hasattr(event, 'movement_y'):
                    y_movement = event.movement_y
                elif hasattr(event, 'delta_y'):
                    y_movement = event.delta_y
                
                # Если получили движение по Y
                if y_movement != 0:
                    direction = "ВВЕРХ" if y_movement < 0 else "ВНИЗ"
                    print(f"Мышь #1 движется {direction} ({abs(y_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse1']['last_move_time'] = current_time
                    
                    # Определяем направление движения
                    if y_movement < -MOVEMENT_THRESHOLD:  # Движение вверх (W)
                        if not movement_state['mouse1']['is_moving_forward']:
                            movement_state['mouse1']['is_moving_forward'] = True
                            movement_state['mouse1']['is_moving_backward'] = False
                            press_w()
                    
                    elif y_movement > MOVEMENT_THRESHOLD:  # Движение вниз (S)
                        if not movement_state['mouse1']['is_moving_backward']:
                            movement_state['mouse1']['is_moving_forward'] = False
                            movement_state['mouse1']['is_moving_backward'] = True
                            press_s()
                    
                    # Запускаем проверку остановки
                    Timer(STOP_DELAY, check_stop_moving).start()
                
            elif device_id == 2:
                # Получаем движение по X (вправо/влево)
                x_movement = 0
                
                # В winrawin движение может быть в разных атрибутах
                # Пробуем получить из разных возможных мест
                if hasattr(event, 'x'):
                    x_movement = event.x
                elif hasattr(event, 'movement_x'):
                    x_movement = event.movement_x
                elif hasattr(event, 'delta_x'):
                    x_movement = event.delta_x
                
                # Если получили движение по X
                if x_movement != 0:
                    direction = "ВПРАВО" if x_movement < 0 else "ВЛЕВО"
                    print(f"Мышь #2 движется {direction} ({abs(x_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse2']['last_move_time'] = current_time
                    
                    # Определяем направление движения
                    if x_movement < -MOVEMENT_THRESHOLD:  # Движение вправо (D)
                        if not movement_state['mouse2']['is_moving_left']:
                            movement_state['mouse2']['is_moving_right'] = False
                            movement_state['mouse2']['is_moving_left'] = True
                            press_a()
                    elif x_movement > MOVEMENT_THRESHOLD:  # Движение влево (A)
                        if not movement_state['mouse2']['is_moving_right']:
                            movement_state['mouse2']['is_moving_right'] = True
                            movement_state['mouse2']['is_moving_left'] = False
                            press_d()
                    
                    # Запускаем проверку остановки
                    Timer(STOP_DELAY, check_stop_moving).start()

def main():
    print("=" * 60)
    print("🖱️  Управление движением вперёд/назад через мышь")
    print("=" * 60)
    print("Программа будет:")
    print("  - Движение мыши 1 ВВЕРХ → Нажимать W (вперёд)")
    print("  - Движение мыши 1 ВНИЗ  → Нажимать S (назад)")
    print("  - Движение мыши 2 ВПРАВО → Нажимать D (вправо)")
    print("  - Движение мыши 2 ВЛЕВО  → Нажимать A (влево)")
    print("  - Остановка мыши → Отпускать клавиши")
    print("  - Вторая мыша игнорируется\n")
    
    print("⚠️  Запустите программу от имени администратора!\n")
    
    # Создаём скрытое окно для получения событий
    root = tk.Tk()
    root.title("Управление для Unity")
    root.geometry("400x150")
    
    # Добавляем информационную метку в окно
    info_label = tk.Label(root, 
                         text="Управление движением:\n"
                              "⬆️ Мышь вверх = W \n"
                              "⬇️ Мышь вниз = S \n"
                              "⬆️ Мышь вправо = D \n"
                              "⬇️ Мышь влево = A \n"
                              "Остановка мыши = стоп",
                         font=("Arial", 10),
                         justify="left")
    info_label.pack(pady=20)
    
    status_label = tk.Label(root, text="Статус: Ожидание движения", fg="blue")
    status_label.pack()
    
    root.iconify()  # Сворачиваем окно
    
    # Список мышей
    mice_handles = list_mice_devices()
    
    if len(mice_handles) < 2:
        print("\n⚠️  Найдена только одна мышь!")
        print("   Подключите вторую мышь для полноценной работы.\n")
    
    print("\n✅ Программа запущена!")
    print("\n   Закройте окно программы для выхода\n")
    
    # Функция обновления статуса в окне
    def update_status():
        if movement_state['mouse1']['w_pressed']:
            status_label.config(text="Статус: Движение ВПЕРЁД (W нажата)", fg="green")
        elif movement_state['mouse1']['s_pressed']:
            status_label.config(text="Статус: Движение НАЗАД (S нажата)", fg="orange")
        elif movement_state['mouse2']['d_pressed']:
            status_label.config(text="Статус: Движение ВПРАВО (D нажата)", fg="green")
        elif movement_state['mouse2']['a_pressed']:
            status_label.config(text="Статус: Движение ВЛЕВО (A нажата)", fg="orange")
        else:
            status_label.config(text="Статус: Ожидание движения", fg="blue")
        root.after(100, update_status)
    
    update_status()
    
    # Подключаем обработчик событий мыши
    winrawin.hook_raw_input_for_window(root.winfo_id(), handle_mouse_event)
    
    try:
        root.mainloop()
    except KeyboardInterrupt:
        pass
    finally:
        release_all()
        print("\n\n👋 Программа завершена")

if __name__ == "__main__":
    # Проверка прав администратора
    import ctypes
    import sys
    
    def is_admin():
        try:
            return ctypes.windll.shell32.IsUserAnAdmin()
        except:
            return False
    
    if not is_admin():
        print("❌ ОШИБКА: Программа должна быть запущена от имени администратора!")
        print("   Нажмите правой кнопкой на файл -> 'Запуск от имени администратора'")
        input("\nНажмите Enter для выхода...")
        sys.exit()
    
    main()

# Версия 2 (только в выбранном окне):

In [ ]:
import winrawin
import tkinter as tk
import keyboard
import time
from threading import Timer
import pygetwindow as gw

# === НАСТРОЙКИ ===
TARGET_WINDOW = "proc-maze-tutorial"  # Сюда впишите название вашего окна
# =================

# Словарь для хранения информации об устройствах
device_map = {}
device_counter = 1

# Переменные для отслеживания состояния движения
movement_state = {
    'mouse1': {
        'is_moving_forward': False,
        'is_moving_backward': False,
        'last_move_time': 0,
        'w_pressed': False,
        's_pressed': False
    },
    'mouse2': {
        'is_moving_right': False,
        'is_moving_left': False,
        'last_move_time': 0,
        'd_pressed': False,
        'a_pressed': False
    }
}

# Таймер для определения остановки движения
STOP_DELAY = 0.15
MOVEMENT_THRESHOLD = 3

def is_target_window_active():
    """Проверяет, активно ли целевое окно"""
    try:
        windows = gw.getWindowsWithTitle(TARGET_WINDOW)
        if not windows:
            return False
        
        target_window = windows[0]
        
        if target_window.isMinimized:
            return False
        
        active_window = gw.getActiveWindow()
        return active_window and active_window.title == TARGET_WINDOW
    except:
        return False

def list_mice_devices():
    """Выводит список всех подключенных мышей"""
    global device_map, device_counter
    
    print("🔍 Поиск подключенных мышей...\n")
    
    devices = winrawin.list_devices()
    mice_handles = []
    
    for device in devices:
        if isinstance(device, winrawin.Mouse):
            print(f"🖱️ Мышь #{device_counter}:")
            print(f"   Тип: {device.mouse_type}")
            print(f"   Handle: {device.handle}\n")
            
            device_map[device.handle] = {
                'id': device_counter,
                'type': device.mouse_type
            }
            
            mice_handles.append(device.handle)
            device_counter += 1
    
    return mice_handles

def press_w():
    """Нажимает и удерживает клавишу W (вперёд)"""
    if not movement_state['mouse1']['w_pressed'] and is_target_window_active():
        if movement_state['mouse1']['s_pressed']:
            keyboard.release('s')
            movement_state['mouse1']['s_pressed'] = False
        
        keyboard.press('w')
        movement_state['mouse1']['w_pressed'] = True
        print("▶️ W НАЖАТА (движение ВПЕРЁД)")

def press_s():
    """Нажимает и удерживает клавишу S (назад)"""
    if not movement_state['mouse1']['s_pressed'] and is_target_window_active():
        if movement_state['mouse1']['w_pressed']:
            keyboard.release('w')
            movement_state['mouse1']['w_pressed'] = False
        
        keyboard.press('s')
        movement_state['mouse1']['s_pressed'] = True
        print("▶️ S НАЖАТА (движение НАЗАД)")

def press_d():
    """Нажимает и удерживает клавишу D (вправо)"""
    if not movement_state['mouse2']['d_pressed'] and is_target_window_active():
        if movement_state['mouse2']['a_pressed']:
            keyboard.release('a')
            movement_state['mouse2']['a_pressed'] = False
        
        keyboard.press('d')
        movement_state['mouse2']['d_pressed'] = True
        print("▶️ D НАЖАТА (движение ВПРАВО)")

def press_a():
    """Нажимает и удерживает клавишу A (влево)"""
    if not movement_state['mouse2']['a_pressed'] and is_target_window_active():
        if movement_state['mouse2']['d_pressed']:
            keyboard.release('d')
            movement_state['mouse2']['d_pressed'] = False
        
        keyboard.press('a')
        movement_state['mouse2']['a_pressed'] = True
        print("▶️ A НАЖАТА (движение ВЛЕВО)")

def release_mouse1():
    """Отпускает все клавиши первой мыши"""
    if movement_state['mouse1']['w_pressed']:
        keyboard.release('w')
        movement_state['mouse1']['w_pressed'] = False
        print("⏹️ W ОТПУЩЕНА")
    
    if movement_state['mouse1']['s_pressed']:
        keyboard.release('s')
        movement_state['mouse1']['s_pressed'] = False
        print("⏹️ S ОТПУЩЕНА")

def release_mouse2():
    """Отпускает все клавиши второй мыши"""
    if movement_state['mouse2']['d_pressed']:
        keyboard.release('d')
        movement_state['mouse2']['d_pressed'] = False
        print("⏹️ D ОТПУЩЕНА")
    
    if movement_state['mouse2']['a_pressed']:
        keyboard.release('a')
        movement_state['mouse2']['a_pressed'] = False
        print("⏹️ A ОТПУЩЕНА")

def release_all():
    """Отпускает все клавиши"""
    release_mouse1()
    release_mouse2()

def check_stop_moving():
    """Проверяет, нужно ли отпустить клавиши из-за остановки мыши"""
    current_time = time.time()
    
    if current_time - movement_state['mouse1']['last_move_time'] > STOP_DELAY:
        movement_state['mouse1']['is_moving_forward'] = False
        movement_state['mouse1']['is_moving_backward'] = False
        release_mouse1()
    
    if current_time - movement_state['mouse2']['last_move_time'] > STOP_DELAY:
        movement_state['mouse2']['is_moving_right'] = False
        movement_state['mouse2']['is_moving_left'] = False
        release_mouse2()

def handle_mouse_event(event: winrawin.RawInputEvent):
    """Обрабатывает события от мышей"""
    
    if event.event_type == 'move':
        device_handle = event.device.handle
        
        if device_handle in device_map:
            device_id = device_map[device_handle]['id']
            
            if device_id == 1:
                # Получаем движение по Y (вверх/вниз)
                y_movement = 0
                
                if hasattr(event, 'y'):
                    y_movement = event.y
                elif hasattr(event, 'movement_y'):
                    y_movement = event.movement_y
                elif hasattr(event, 'delta_y'):
                    y_movement = event.delta_y
                
                if y_movement != 0:
                    direction = "ВВЕРХ" if y_movement < 0 else "ВНИЗ"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #1 движется {direction} ({abs(y_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse1']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if y_movement < -MOVEMENT_THRESHOLD:  # Движение вверх (W)
                            if not movement_state['mouse1']['is_moving_forward']:
                                movement_state['mouse1']['is_moving_forward'] = True
                                movement_state['mouse1']['is_moving_backward'] = False
                                press_w()
                        
                        elif y_movement > MOVEMENT_THRESHOLD:  # Движение вниз (S)
                            if not movement_state['mouse1']['is_moving_backward']:
                                movement_state['mouse1']['is_moving_forward'] = False
                                movement_state['mouse1']['is_moving_backward'] = True
                                press_s()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()
                
            elif device_id == 2:
                # Получаем движение по X (вправо/влево)
                x_movement = 0
                
                if hasattr(event, 'x'):
                    x_movement = event.x
                elif hasattr(event, 'movement_x'):
                    x_movement = event.movement_x
                elif hasattr(event, 'delta_x'):
                    x_movement = event.delta_x
                
                if x_movement != 0:
                    direction = "ВПРАВО" if x_movement > 0 else "ВЛЕВО"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #2 движется {direction} ({abs(x_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse2']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if x_movement > MOVEMENT_THRESHOLD:  # Движение вправо (D)
                            if not movement_state['mouse2']['is_moving_right']:
                                movement_state['mouse2']['is_moving_right'] = True
                                movement_state['mouse2']['is_moving_left'] = False
                                press_d()
                        
                        elif x_movement < -MOVEMENT_THRESHOLD:  # Движение влево (A)
                            if not movement_state['mouse2']['is_moving_left']:
                                movement_state['mouse2']['is_moving_right'] = False
                                movement_state['mouse2']['is_moving_left'] = True
                                press_a()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()

def main():
    print("=" * 60)
    print("🖱️  Управление движением в окне:", TARGET_WINDOW)
    print("=" * 60)
    print("Программа будет:")
    print("  - Мышь #1: ВВЕРХ → W, ВНИЗ → S")
    print("  - Мышь #2: ВПРАВО → D, ВЛЕВО → A")
    print(f"  - Клавиши работают ТОЛЬКО в окне: '{TARGET_WINDOW}'")
    print("  - Остановка мыши → Отпускать клавиши\n")
    
    print("⚠️  Запустите программу от имени администратора!\n")
    
    # Создаём скрытое окно для получения событий
    root = tk.Tk()
    root.title("Управление для Unity")
    root.geometry("400x150")
    
    # Добавляем информационную метку в окно
    info_label = tk.Label(root, 
                         text=f"Управление движением:\n"
                              f"Целевое окно: {TARGET_WINDOW}\n"
                              f"⬆️ Мышь 1 вверх = W\n"
                              f"⬇️ Мышь 1 вниз = S\n"
                              f"➡️ Мышь 2 вправо = D\n"
                              f"⬅️ Мышь 2 влево = A\n"
                              f"Остановка мыши = стоп",
                         font=("Arial", 9),
                         justify="left")
    info_label.pack(pady=10)
    
    status_label = tk.Label(root, text="Статус: Ожидание движения", fg="blue")
    status_label.pack()
    
    root.iconify()  # Сворачиваем окно
    
    # Список мышей
    mice_handles = list_mice_devices()
    
    if len(mice_handles) < 2:
        print("\n⚠️  Найдена только одна мышь!")
        print("   Подключите вторую мышь для полноценной работы.\n")
    
    print("\n✅ Программа запущена!")
    print(f"   Клавиши будут работать только в окне: '{TARGET_WINDOW}'")
    print("   🎯 - окно активно, ⏸️ - окно не активно\n")
    print("   Закройте окно программы для выхода\n")
    
    # Функция обновления статуса в окне
    def update_status():
        if not is_target_window_active():
            status_label.config(text="Статус: Окно не активно (⏸️)", fg="red")
        elif movement_state['mouse1']['w_pressed']:
            status_label.config(text="Статус: Движение ВПЕРЁД (W нажата)", fg="green")
        elif movement_state['mouse1']['s_pressed']:
            status_label.config(text="Статус: Движение НАЗАД (S нажата)", fg="orange")
        elif movement_state['mouse2']['d_pressed']:
            status_label.config(text="Статус: Движение ВПРАВО (D нажата)", fg="green")
        elif movement_state['mouse2']['a_pressed']:
            status_label.config(text="Статус: Движение ВЛЕВО (A нажата)", fg="orange")
        else:
            status_label.config(text="Статус: Ожидание движения", fg="blue")
        root.after(100, update_status)
    
    update_status()
    
    # Подключаем обработчик событий мыши
    winrawin.hook_raw_input_for_window(root.winfo_id(), handle_mouse_event)
    
    try:
        root.mainloop()
    except KeyboardInterrupt:
        pass
    finally:
        release_all()
        print("\n\n👋 Программа завершена")

if __name__ == "__main__":
    # Проверка прав администратора
    import ctypes
    import sys
    
    def is_admin():
        try:
            return ctypes.windll.shell32.IsUserAnAdmin()
        except:
            return False
    
    if not is_admin():
        print("❌ ОШИБКА: Программа должна быть запущена от имени администратора!")
        print("   Нажмите правой кнопкой на файл -> 'Запуск от имени администратора'")
        input("\nНажмите Enter для выхода...")
        sys.exit()
    
    main()

# Версия 3 (каллибровка мышей), должны блокироваться мыши в окне, но не блокируются:

In [ ]:
import winrawin
import tkinter as tk
import keyboard
import time
from threading import Timer
import pygetwindow as gw

# === НАСТРОЙКИ ===
TARGET_WINDOW = "proc-maze-tutorial"  # Сюда впишите название вашего окна
# =================

# Словарь для хранения информации об устройствах
device_map = {}
device_counter = 1
grabbed_handles = []

# Переменные для калибровки
calibration_mode = False
calibration_step = 0
calibration_mice = {}  # {handle: {'id': номер}}
waiting_for_mouse = None
calibration_complete = False

# Переменные для отслеживания состояния движения
movement_state = {
    'mouse1': {
        'is_moving_forward': False,
        'is_moving_backward': False,
        'last_move_time': 0,
        'w_pressed': False,
        's_pressed': False
    },
    'mouse2': {
        'is_moving_right': False,
        'is_moving_left': False,
        'last_move_time': 0,
        'd_pressed': False,
        'a_pressed': False
    }
}

# Таймер для определения остановки движения
STOP_DELAY = 0.15
MOVEMENT_THRESHOLD = 3


def start_calibration():
    """Запускает режим калибровки мышей"""
    global calibration_mode, calibration_step, calibration_mice, waiting_for_mouse, calibration_complete
    
    print("\n" + "="*50)
    print("🔧 РЕЖИМ КАЛИБРОВКИ МЫШЕЙ")
    print("="*50)
    print("Сейчас нужно настроить мыши:")
    print("  1. Нажмите ЛЮБУЮ КНОПКУ на мыши, которая будет УПРАВЛЯТЬ ДВИЖЕНИЕМ (W/S)")
    print("  2. Нажмите ЛЮБУЮ КНОПКУ на мыши, которая будет ПОВОРАЧИВАТЬ (A/D)")
    print("  3. Нажимайте ЛЮБЫЕ КНОПКИ на остальных мышах (будут игнорироваться)")
    print("\n🖱️ Жду нажатие на МЫШЬ #1 (движение W/S)...")
    
    calibration_mode = True
    calibration_step = 1
    calibration_mice = {}
    waiting_for_mouse = 1
    calibration_complete = False

def complete_calibration():
    """Завершает калибровку"""
    global calibration_mode, calibration_step, calibration_mice, waiting_for_mouse, device_map, calibration_complete
    
    print("\n" + "="*50)
    print("✅ КАЛИБРОВКА ЗАВЕРШЕНА")
    print("="*50)
    print("Настройка мышей:")
    
    # Очищаем старую device_map
    device_map.clear()
    
    # Заполняем новую device_map на основе калибровки
    mouse_count = 1
    for handle, info in calibration_mice.items():
        if mouse_count == 1:
            device_id = 1
            role = "ДВИЖЕНИЕ (W/S)"
        elif mouse_count == 2:
            device_id = 2
            role = "ПОВОРОТ (A/D)"
        else:
            device_id = mouse_count
            role = "ИГНОРИРУЕТСЯ"
        
        device_map[handle] = {
            'id': device_id,
            'type': 'mouse'
        }
        print(f"  🖱️ Мышь #{device_id}: {role}")
        mouse_count += 1
    
    print("\n🎮 Теперь можно использовать мыши!")
    print("-"*50)
    
    calibration_mode = False
    calibration_step = 0
    calibration_mice = {}
    waiting_for_mouse = None
    calibration_complete = True

def handle_calibration_event(event: winrawin.RawInputEvent):
    """Обрабатывает события во время калибровки"""
    global calibration_step, calibration_mice, waiting_for_mouse
    
    if event.event_type == 'down':  # Нажатие кнопки
        device_handle = event.device.handle
        
        # Проверяем, не нажимали ли уже эту мышь
        if device_handle in calibration_mice:
            print(f"⚠️  Эта мышь уже настроена как #{calibration_mice[device_handle]}")
            return
        
        if calibration_step == 1:  # Ждём первую мышь
            calibration_mice[device_handle] = 1
            print(f"  ✅ Мышь #1 настроена")
            print("\n🖱️ Теперь нажмите любую кнопку на МЫШЬ #2 (поворот A/D)...")
            calibration_step = 2
            waiting_for_mouse = 2
            
        elif calibration_step == 2:  # Ждём вторую мышь
            calibration_mice[device_handle] = 2
            print(f"  ✅ Мышь #2 настроена")
            print("\n🖱️ Теперь нажимайте любые кнопки на остальных мышах...")
            print("   (Они будут игнорироваться в игре)")
            print("   Для завершения калибровки нажмите кнопку 'Завершить' в окне")
            calibration_step = 3
            waiting_for_mouse = "остальные"
            
        elif calibration_step == 3:  # Остальные мыши
            # Присваиваем следующий доступный ID
            next_id = len(calibration_mice) + 1
            calibration_mice[device_handle] = next_id
            print(f"  📱 Мышь #{next_id} будет игнорироваться")
            
def grab_mice():
    """Блокирует все мыши через Raw Input API"""
    global grabbed_handles
    
    if grabbed_handles:  # Уже заблокированы
        return True
    
    print("\n🔒 Блокировка мышей...")
    # В winrawin захват делается через хук, а не через Device
    # Просто запоминаем, какие мыши нужно блокировать
    for handle, info in device_map.items():
        if info['type'] == 'mouse':  # Блокируем только мыши
            grabbed_handles.append(handle)
            print(f"   ✅ Мышь #{info['id']} будет заблокирована")
    
    if grabbed_handles:
        print(f"✅ Заблокировано мышей: {len(grabbed_handles)}\n")
        return True
    return False

def ungrab_mice():
    """Разблокирует все мыши"""
    global grabbed_handles
    
    if not grabbed_handles:  # Уже разблокированы
        return
    
    print("\n🔓 Разблокировка мышей...")
    for handle in grabbed_handles[:]:
        if handle in device_map:
            print(f"   ✅ Мышь #{device_map[handle]['id']} разблокирована")
    
    grabbed_handles.clear()
    print("✅ Все мыши разблокированы\n")

def is_target_window_active():
    """Проверяет, активно ли целевое окно"""
    try:
        windows = gw.getWindowsWithTitle(TARGET_WINDOW)
        if not windows:
            return False
        
        target_window = windows[0]
        
        # Проверяем, что окно существует и не свёрнуто
        if not target_window or target_window.isMinimized:
            return False
        
        # Получаем активное окно
        active_window = gw.getActiveWindow()
        if not active_window:
            return False
        
        # Сравниваем заголовки
        return active_window.title == TARGET_WINDOW
    except Exception as e:
        print(f"Ошибка проверки окна: {e}")
        return False

def list_mice_devices():
    """Выводит список всех подключенных мышей"""
    global device_map, device_counter
    
    print("🔍 Поиск подключенных мышей...\n")
    
    devices = winrawin.list_devices()
    mice_handles = []
    device_counter = 1  # Сбрасываем счетчик
    
    for device in devices:
        # Проверяем, что это действительно мышь
        if isinstance(device, winrawin.Mouse):
            # Дополнительная проверка по типу устройства
            if hasattr(device, 'mouse_type') and device.mouse_type == 'mouse':
                print(f"🖱️ Мышь #{device_counter}:")
                print(f"   Handle: {device.handle}")
                print(f"   Тип: {device.mouse_type}\n")
                
                device_map[device.handle] = {
                    'id': device_counter,
                    'type': 'mouse'
                }
                
                mice_handles.append(device.handle)
                device_counter += 1
            else:
                print(f"📱 Не мышь (устройство #{device_counter}): {getattr(device, 'mouse_type', 'неизвестно')}")
                device_counter += 1
        else:
            print(f"📱 Не мышь (устройство #{device_counter}): {type(device).__name__}")
            device_counter += 1
    
    return mice_handles

def press_w():
    """Нажимает и удерживает клавишу W (вперёд)"""
    if not movement_state['mouse1']['w_pressed'] and is_target_window_active():
        if movement_state['mouse1']['s_pressed']:
            keyboard.release('s')
            movement_state['mouse1']['s_pressed'] = False
        
        keyboard.press('w')
        movement_state['mouse1']['w_pressed'] = True
        print("▶️ W НАЖАТА (движение ВПЕРЁД)")

def press_s():
    """Нажимает и удерживает клавишу S (назад)"""
    if not movement_state['mouse1']['s_pressed'] and is_target_window_active():
        if movement_state['mouse1']['w_pressed']:
            keyboard.release('w')
            movement_state['mouse1']['w_pressed'] = False
        
        keyboard.press('s')
        movement_state['mouse1']['s_pressed'] = True
        print("▶️ S НАЖАТА (движение НАЗАД)")

def press_d():
    """Нажимает и удерживает клавишу D (вправо)"""
    if not movement_state['mouse2']['d_pressed'] and is_target_window_active():
        if movement_state['mouse2']['a_pressed']:
            keyboard.release('a')
            movement_state['mouse2']['a_pressed'] = False
        
        keyboard.press('d')
        movement_state['mouse2']['d_pressed'] = True
        print("▶️ D НАЖАТА (движение ВПРАВО)")

def press_a():
    """Нажимает и удерживает клавишу A (влево)"""
    if not movement_state['mouse2']['a_pressed'] and is_target_window_active():
        if movement_state['mouse2']['d_pressed']:
            keyboard.release('d')
            movement_state['mouse2']['d_pressed'] = False
        
        keyboard.press('a')
        movement_state['mouse2']['a_pressed'] = True
        print("▶️ A НАЖАТА (движение ВЛЕВО)")

def release_mouse1():
    """Отпускает все клавиши первой мыши"""
    if movement_state['mouse1']['w_pressed']:
        keyboard.release('w')
        movement_state['mouse1']['w_pressed'] = False
        print("⏹️ W ОТПУЩЕНА")
    
    if movement_state['mouse1']['s_pressed']:
        keyboard.release('s')
        movement_state['mouse1']['s_pressed'] = False
        print("⏹️ S ОТПУЩЕНА")

def release_mouse2():
    """Отпускает все клавиши второй мыши"""
    if movement_state['mouse2']['d_pressed']:
        keyboard.release('d')
        movement_state['mouse2']['d_pressed'] = False
        print("⏹️ D ОТПУЩЕНА")
    
    if movement_state['mouse2']['a_pressed']:
        keyboard.release('a')
        movement_state['mouse2']['a_pressed'] = False
        print("⏹️ A ОТПУЩЕНА")

def release_all():
    """Отпускает все клавиши"""
    release_mouse1()
    release_mouse2()

def check_stop_moving():
    """Проверяет, нужно ли отпустить клавиши из-за остановки мыши"""
    current_time = time.time()
    
    if current_time - movement_state['mouse1']['last_move_time'] > STOP_DELAY:
        movement_state['mouse1']['is_moving_forward'] = False
        movement_state['mouse1']['is_moving_backward'] = False
        release_mouse1()
    
    if current_time - movement_state['mouse2']['last_move_time'] > STOP_DELAY:
        movement_state['mouse2']['is_moving_right'] = False
        movement_state['mouse2']['is_moving_left'] = False
        release_mouse2()

def handle_mouse_event(event: winrawin.RawInputEvent):
    """Обрабатывает события от мышей"""
    # Если режим калибровки - обрабатываем по-особому
    if calibration_mode:
        handle_calibration_event(event)
        return
        
    if event.event_type == 'move':
        device_handle = event.device.handle
        
        # Проверяем, что это мышь из нашего списка
        if device_handle not in device_map:
            return
        
        # Проверяем активность окна и управляем "блокировкой"
        if is_target_window_active():
            if not grabbed_handles:  # Если окно активно, но мыши не "заблокированы"
                grab_mice()
        else:
            if grabbed_handles:  # Если окно не активно, но мыши "заблокированы"
                ungrab_mice()
                release_all()  # Отпускаем клавиши при выходе из окна
        
        device_id = device_map[device_handle]['id']
    
    if event.event_type == 'move':
        device_handle = event.device.handle
        
        if device_handle in device_map:
            device_id = device_map[device_handle]['id']
            
            if device_id == 1:
                # Получаем движение по Y (вверх/вниз)
                y_movement = 0
                
                if hasattr(event, 'y'):
                    y_movement = event.y
                elif hasattr(event, 'movement_y'):
                    y_movement = event.movement_y
                elif hasattr(event, 'delta_y'):
                    y_movement = event.delta_y
                
                if y_movement != 0:
                    direction = "ВВЕРХ" if y_movement < 0 else "ВНИЗ"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #1 движется {direction} ({abs(y_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse1']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if y_movement < -MOVEMENT_THRESHOLD:  # Движение вверх (W)
                            if not movement_state['mouse1']['is_moving_forward']:
                                movement_state['mouse1']['is_moving_forward'] = True
                                movement_state['mouse1']['is_moving_backward'] = False
                                press_w()
                        
                        elif y_movement > MOVEMENT_THRESHOLD:  # Движение вниз (S)
                            if not movement_state['mouse1']['is_moving_backward']:
                                movement_state['mouse1']['is_moving_forward'] = False
                                movement_state['mouse1']['is_moving_backward'] = True
                                press_s()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()
                
            elif device_id == 2:
                # Получаем движение по X (вправо/влево)
                x_movement = 0
                
                if hasattr(event, 'x'):
                    x_movement = event.x
                elif hasattr(event, 'movement_x'):
                    x_movement = event.movement_x
                elif hasattr(event, 'delta_x'):
                    x_movement = event.delta_x
                
                if x_movement != 0:
                    direction = "ВПРАВО" if x_movement > 0 else "ВЛЕВО"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #2 движется {direction} ({abs(x_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse2']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if x_movement > MOVEMENT_THRESHOLD:  # Движение вправо (D)
                            if not movement_state['mouse2']['is_moving_right']:
                                movement_state['mouse2']['is_moving_right'] = True
                                movement_state['mouse2']['is_moving_left'] = False
                                press_d()
                        
                        elif x_movement < -MOVEMENT_THRESHOLD:  # Движение влево (A)
                            if not movement_state['mouse2']['is_moving_left']:
                                movement_state['mouse2']['is_moving_right'] = False
                                movement_state['mouse2']['is_moving_left'] = True
                                press_a()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()

def main():
    print("=" * 60)
    print("🖱️  Управление движением в окне:", TARGET_WINDOW)
    print("=" * 60)
    
    print("⚠️  Запустите программу от имени администратора!\n")
    
    # Создаём окно для получения событий
    root = tk.Tk()
    root.title("Управление для Unity")
    root.geometry("450x250")
    
    # Фрейм для информации
    info_frame = tk.Frame(root)
    info_frame.pack(pady=10)
    
    info_label = tk.Label(info_frame, 
                         text=f"Целевое окно: {TARGET_WINDOW}\n\n"
                              f"⬆️ Мышь 1 вверх = W\n"
                              f"⬇️ Мышь 1 вниз = S\n"
                              f"➡️ Мышь 2 вправо = D\n"
                              f"⬅️ Мышь 2 влево = A",
                         font=("Arial", 9),
                         justify="left")
    info_label.pack()
    
    # Статус
    status_label = tk.Label(root, text="Статус: Ожидание калибровки", fg="blue")
    status_label.pack(pady=5)
    
    # Фрейм для кнопок
    button_frame = tk.Frame(root)
    button_frame.pack(pady=10)
    
    def start_calibration_from_gui():
        start_calibration()
        status_label.config(text="Статус: РЕЖИМ КАЛИБРОВКИ", fg="orange")
        calibrate_btn.config(state="disabled")
        complete_btn.config(state="normal")
    
    def complete_calibration_from_gui():
        global calibration_complete
        if calibration_mode and calibration_step >= 2:
            complete_calibration()
            status_label.config(text="Статус: Калибровка завершена", fg="green")
            calibrate_btn.config(state="normal")
            complete_btn.config(state="disabled")
            
            # Показываем найденные мыши
            mice_text = "Найденные мыши:\n"
            for handle, info in device_map.items():
                role = "ДВИЖЕНИЕ" if info['id'] == 1 else "ПОВОРОТ" if info['id'] == 2 else "ИГНОР"
                mice_text += f"  Мышь #{info['id']}: {role}\n"
            info_label.config(text=info_label.cget("text") + "\n\n" + mice_text)
    
    calibrate_btn = tk.Button(button_frame, text="🔧 Начать калибровку", 
                              command=start_calibration_from_gui, bg="lightblue")
    calibrate_btn.pack(side="left", padx=5)
    
    complete_btn = tk.Button(button_frame, text="✅ Завершить калибровку", 
                             command=complete_calibration_from_gui, 
                             bg="lightgreen", state="disabled")
    complete_btn.pack(side="left", padx=5)
    
    # Список мышей для информации
    mice_handles = list_mice_devices()
    
    if len(mice_handles) < 2:
        print("\n⚠️  Найдена только одна мышь!")
        print("   Подключите вторую мышь для полноценной работы.\n")
    
    print("\n✅ Программа запущена!")
    print("   Используйте кнопки в окне для калибровки мышей")
    print(f"   Клавиши будут работать только в окне: '{TARGET_WINDOW}'")
    print("   🎯 - окно активно, ⏸️ - окно не активно\n")
    
    # Функция обновления статуса
    def update_status():
        if calibration_mode:
            if calibration_step == 1:
                status_label.config(text="Статус: Калибровка - ждём мышь #1", fg="orange")
            elif calibration_step == 2:
                status_label.config(text="Статус: Калибровка - ждём мышь #2", fg="orange")
            elif calibration_step == 3:
                status_label.config(text="Статус: Калибровка - ждём остальные мыши", fg="orange")
        elif not is_target_window_active():
            status_label.config(text="Статус: Окно не активно (⏸️)", fg="red")
        elif movement_state['mouse1']['w_pressed']:
            status_label.config(text="Статус: Движение ВПЕРЁД (W)", fg="green")
        elif movement_state['mouse1']['s_pressed']:
            status_label.config(text="Статус: Движение НАЗАД (S)", fg="orange")
        elif movement_state['mouse2']['d_pressed']:
            status_label.config(text="Статус: Движение ВПРАВО (D)", fg="green")
        elif movement_state['mouse2']['a_pressed']:
            status_label.config(text="Статус: Движение ВЛЕВО (A)", fg="orange")
        else:
            status_label.config(text="Статус: Ожидание движения", fg="blue")
        root.after(100, update_status)
    
    update_status()
    
    # Подключаем обработчик событий мыши
    winrawin.hook_raw_input_for_window(root.winfo_id(), handle_mouse_event)
    
    try:
        root.mainloop()
    except KeyboardInterrupt:
        pass
    finally:
        release_all()
        print("\n\n👋 Программа завершена")
    
main()

# Версия 4 (только вертикальные движения)

In [ ]:
import winrawin
import tkinter as tk
import keyboard
import time
from threading import Timer
import pygetwindow as gw

# === НАСТРОЙКИ ===
TARGET_WINDOW = "proc-maze-tutorial"  # Сюда впишите название вашего окна
# =================

# Словарь для хранения информации об устройствах
device_map = {}
device_counter = 1

# Переменные для отслеживания состояния движения
movement_state = {
    'mouse1': {
        'is_moving_forward': False,
        'is_moving_backward': False,
        'last_move_time': 0,
        'w_pressed': False,
        's_pressed': False
    },
    'mouse2': {
        'is_moving_right': False,
        'is_moving_left': False,
        'last_move_time': 0,
        'd_pressed': False,
        'a_pressed': False
    }
}

# Таймер для определения остановки движения
STOP_DELAY = 0.15
MOVEMENT_THRESHOLD = 3

def is_target_window_active():
    """Проверяет, активно ли целевое окно"""
    try:
        windows = gw.getWindowsWithTitle(TARGET_WINDOW)
        if not windows:
            return False
        
        target_window = windows[0]
        
        if target_window.isMinimized:
            return False
        
        active_window = gw.getActiveWindow()
        return active_window and active_window.title == TARGET_WINDOW
    except:
        return False

def list_mice_devices():
    """Выводит список всех подключенных мышей"""
    global device_map, device_counter
    
    print("🔍 Поиск подключенных мышей...\n")
    
    devices = winrawin.list_devices()
    mice_handles = []
    
    for device in devices:
        if isinstance(device, winrawin.Mouse):
            print(f"🖱️ Мышь #{device_counter}:")
            print(f"   Тип: {device.mouse_type}")
            print(f"   Handle: {device.handle}\n")
            
            device_map[device.handle] = {
                'id': device_counter,
                'type': device.mouse_type
            }
            
            mice_handles.append(device.handle)
            device_counter += 1
    
    return mice_handles

def press_w():
    """Нажимает и удерживает клавишу W (вперёд)"""
    if not movement_state['mouse1']['w_pressed'] and is_target_window_active():
        if movement_state['mouse1']['s_pressed']:
            keyboard.release('s')
            movement_state['mouse1']['s_pressed'] = False
        
        keyboard.press('w')
        movement_state['mouse1']['w_pressed'] = True
        print("▶️ W НАЖАТА (движение ВПЕРЁД)")

def press_s():
    """Нажимает и удерживает клавишу S (назад)"""
    if not movement_state['mouse1']['s_pressed'] and is_target_window_active():
        if movement_state['mouse1']['w_pressed']:
            keyboard.release('w')
            movement_state['mouse1']['w_pressed'] = False
        
        keyboard.press('s')
        movement_state['mouse1']['s_pressed'] = True
        print("▶️ S НАЖАТА (движение НАЗАД)")

def press_d():
    """Нажимает и удерживает клавишу D (вправо)"""
    if not movement_state['mouse2']['d_pressed'] and is_target_window_active():
        if movement_state['mouse2']['a_pressed']:
            keyboard.release('a')
            movement_state['mouse2']['a_pressed'] = False
        
        keyboard.press('d')
        movement_state['mouse2']['d_pressed'] = True
        print("▶️ D НАЖАТА (движение ВПРАВО)")

def press_a():
    """Нажимает и удерживает клавишу A (влево)"""
    if not movement_state['mouse2']['a_pressed'] and is_target_window_active():
        if movement_state['mouse2']['d_pressed']:
            keyboard.release('d')
            movement_state['mouse2']['d_pressed'] = False
        
        keyboard.press('a')
        movement_state['mouse2']['a_pressed'] = True
        print("▶️ A НАЖАТА (движение ВЛЕВО)")

def release_mouse1():
    """Отпускает все клавиши первой мыши"""
    if movement_state['mouse1']['w_pressed']:
        keyboard.release('w')
        movement_state['mouse1']['w_pressed'] = False
        print("⏹️ W ОТПУЩЕНА")
    
    if movement_state['mouse1']['s_pressed']:
        keyboard.release('s')
        movement_state['mouse1']['s_pressed'] = False
        print("⏹️ S ОТПУЩЕНА")

def release_mouse2():
    """Отпускает все клавиши второй мыши"""
    if movement_state['mouse2']['d_pressed']:
        keyboard.release('d')
        movement_state['mouse2']['d_pressed'] = False
        print("⏹️ D ОТПУЩЕНА")
    
    if movement_state['mouse2']['a_pressed']:
        keyboard.release('a')
        movement_state['mouse2']['a_pressed'] = False
        print("⏹️ A ОТПУЩЕНА")

def release_all():
    """Отпускает все клавиши"""
    release_mouse1()
    release_mouse2()

def check_stop_moving():
    """Проверяет, нужно ли отпустить клавиши из-за остановки мыши"""
    current_time = time.time()
    
    if current_time - movement_state['mouse1']['last_move_time'] > STOP_DELAY:
        movement_state['mouse1']['is_moving_forward'] = False
        movement_state['mouse1']['is_moving_backward'] = False
        release_mouse1()
    
    if current_time - movement_state['mouse2']['last_move_time'] > STOP_DELAY:
        movement_state['mouse2']['is_moving_right'] = False
        movement_state['mouse2']['is_moving_left'] = False
        release_mouse2()

def handle_mouse_event(event: winrawin.RawInputEvent):
    """Обрабатывает события от мышей"""
    
    if event.event_type == 'move':
        device_handle = event.device.handle
        
        if device_handle in device_map:
            device_id = device_map[device_handle]['id']
            
            if device_id == 1:
                # Получаем движение по Y (вверх/вниз)
                y_movement = 0
                
                if hasattr(event, 'y'):
                    y_movement = event.y
                elif hasattr(event, 'movement_y'):
                    y_movement = event.movement_y
                elif hasattr(event, 'delta_y'):
                    y_movement = event.delta_y
                
                if y_movement != 0:
                    direction = "ВВЕРХ" if y_movement < 0 else "ВНИЗ"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #1 движется {direction} ({abs(y_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse1']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if y_movement < -MOVEMENT_THRESHOLD:  # Движение вверх (W)
                            if not movement_state['mouse1']['is_moving_forward']:
                                movement_state['mouse1']['is_moving_forward'] = True
                                movement_state['mouse1']['is_moving_backward'] = False
                                press_w()
                        
                        elif y_movement > MOVEMENT_THRESHOLD:  # Движение вниз (S)
                            if not movement_state['mouse1']['is_moving_backward']:
                                movement_state['mouse1']['is_moving_forward'] = False
                                movement_state['mouse1']['is_moving_backward'] = True
                                press_s()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()
                
            elif device_id == 2:
                # Получаем движение по X (вправо/влево)
                x_movement = 0
                
                if hasattr(event, 'y'):
                    x_movement = event.y
                elif hasattr(event, 'movement_y'):
                    x_movement = event.movement_y
                elif hasattr(event, 'delta_y'):
                    x_movement = event.delta_y
                
                if x_movement != 0:
                    direction = "ВПРАВО" if x_movement > 0 else "ВЛЕВО"
                    status = "🎯" if is_target_window_active() else "⏸️"
                    print(f"{status} Мышь #2 движется {direction} ({abs(x_movement)} пикс)", end='\r')
                    
                    current_time = time.time()
                    movement_state['mouse2']['last_move_time'] = current_time
                    
                    if is_target_window_active():
                        if x_movement > MOVEMENT_THRESHOLD:  # Движение вправо (D)
                            if not movement_state['mouse2']['is_moving_right']:
                                movement_state['mouse2']['is_moving_right'] = True
                                movement_state['mouse2']['is_moving_left'] = False
                                press_d()
                        
                        elif x_movement < -MOVEMENT_THRESHOLD:  # Движение влево (A)
                            if not movement_state['mouse2']['is_moving_left']:
                                movement_state['mouse2']['is_moving_right'] = False
                                movement_state['mouse2']['is_moving_left'] = True
                                press_a()
                    
                    Timer(STOP_DELAY, check_stop_moving).start()

def main():
    print("=" * 60)
    print("🖱️  Управление движением в окне:", TARGET_WINDOW)
    print("=" * 60)
    print("Программа будет:")
    print("  - Мышь #1: ВВЕРХ → W, ВНИЗ → S")
    print("  - Мышь #2: ВПРАВО → D, ВЛЕВО → A")
    print(f"  - Клавиши работают ТОЛЬКО в окне: '{TARGET_WINDOW}'")
    print("  - Остановка мыши → Отпускать клавиши\n")
    
    print("⚠️  Запустите программу от имени администратора!\n")
    
    # Создаём скрытое окно для получения событий
    root = tk.Tk()
    root.title("Управление для Unity")
    root.geometry("400x150")
    
    # Добавляем информационную метку в окно
    info_label = tk.Label(root, 
                         text=f"Управление движением:\n"
                              f"Целевое окно: {TARGET_WINDOW}\n"
                              f"⬆️ Мышь 1 вверх = W\n"
                              f"⬇️ Мышь 1 вниз = S\n"
                              f"➡️ Мышь 2 вправо = D\n"
                              f"⬅️ Мышь 2 влево = A\n"
                              f"Остановка мыши = стоп",
                         font=("Arial", 9),
                         justify="left")
    info_label.pack(pady=10)
    
    status_label = tk.Label(root, text="Статус: Ожидание движения", fg="blue")
    status_label.pack()
    
    root.iconify()  # Сворачиваем окно
    
    # Список мышей
    mice_handles = list_mice_devices()
    
    if len(mice_handles) < 2:
        print("\n⚠️  Найдена только одна мышь!")
        print("   Подключите вторую мышь для полноценной работы.\n")
    
    print("\n✅ Программа запущена!")
    print(f"   Клавиши будут работать только в окне: '{TARGET_WINDOW}'")
    print("   🎯 - окно активно, ⏸️ - окно не активно\n")
    print("   Закройте окно программы для выхода\n")
    
    # Функция обновления статуса в окне
    def update_status():
        if not is_target_window_active():
            status_label.config(text="Статус: Окно не активно (⏸️)", fg="red")
        elif movement_state['mouse1']['w_pressed']:
            status_label.config(text="Статус: Движение ВПЕРЁД (W нажата)", fg="green")
        elif movement_state['mouse1']['s_pressed']:
            status_label.config(text="Статус: Движение НАЗАД (S нажата)", fg="orange")
        elif movement_state['mouse2']['d_pressed']:
            status_label.config(text="Статус: Движение ВПРАВО (D нажата)", fg="green")
        elif movement_state['mouse2']['a_pressed']:
            status_label.config(text="Статус: Движение ВЛЕВО (A нажата)", fg="orange")
        else:
            status_label.config(text="Статус: Ожидание движения", fg="blue")
        root.after(100, update_status)
    
    update_status()
    
    # Подключаем обработчик событий мыши
    winrawin.hook_raw_input_for_window(root.winfo_id(), handle_mouse_event)
    
    try:
        root.mainloop()
    except KeyboardInterrupt:
        pass
    finally:
        release_all()
        print("\n\n👋 Программа завершена")

if __name__ == "__main__":
    # Проверка прав администратора
    import ctypes
    import sys
    
    def is_admin():
        try:
            return ctypes.windll.shell32.IsUserAnAdmin()
        except:
            return False
    
    if not is_admin():
        print("❌ ОШИБКА: Программа должна быть запущена от имени администратора!")
        print("   Нажмите правой кнопкой на файл -> 'Запуск от имени администратора'")
        input("\nНажмите Enter для выхода...")
        sys.exit()
    
    main()